In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import json
import os
from tqdm import tqdm

In [7]:
# --- 1. CONFIGURAZIONE ---
INPUT_JSONL = "/kaggle/input/data-to-judge/dpo_dataset_ready_v2.jsonl"   
MODEL_PATH = "/kaggle/input/deberta-weights/deberta_judge.pth"             
OUTPUT_JSONL = "dpo_dataset_FILTERED.jsonl"  

MODEL_NAME = "microsoft/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f" Device attivo: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

 Device attivo: cuda


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [8]:
# --- 2. DEFINIZIONE ARCHITETTURA (Uguale al training) ---
class DebertaJudge(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)
        
    def forward(self, input_ids, mask):
        out = self.bert(input_ids, attention_mask=mask)
        return self.regressor(out.last_hidden_state[:, 0, :])

In [9]:
# --- 3. CARICAMENTO MODELLO ---
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Manca il file {MODEL_PATH}!")

if not os.path.exists(INPUT_JSONL):
    raise FileNotFoundError(f"Manca il file {INPUT_JSONL}!")

print("Caricamento Giudice...")
model = DebertaJudge()
try:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
except Exception as e:
    print(f"Errore caricamento pesi: {e}")
    raise e

model.to(DEVICE)
model.eval() 

Caricamento Giudice...


DebertaJudge(
  (bert): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-07, e

In [10]:
# --- 4. FUNZIONE DI SCORING ---
def get_score(text):
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=128, 
        padding=True
    ).to(DEVICE)
    
    with torch.no_grad():
        score = model(inputs['input_ids'], inputs['attention_mask'])
    return score.item()

In [11]:
# --- 5. ESECUZIONE FILTRO ---
print(f"Analisi coppie in corso su: {INPUT_JSONL}")

valid_pairs = 0
skipped_pairs = 0
dataset_buffer = []

with open(INPUT_JSONL, 'r') as f:
    lines = f.readlines()

print(f"Totale righe da analizzare: {len(lines)}")

for line in tqdm(lines):
    try:
        entry = json.loads(line)
        
        # Estrarre testo (gestione robusta, lista vs stringa)
        # Zephyr DPO format: "chosen": [{"role":..., "content": "..."}]
        
        if isinstance(entry['chosen'], list):
            chosen_text = entry['chosen'][-1]['content']
            rejected_text = entry['rejected'][-1]['content']
        else:
            chosen_text = entry['chosen']
            rejected_text = entry['rejected']
        
        # Valutazione
        score_chosen = get_score(chosen_text)
        score_rejected = get_score(rejected_text)
        
        # Accettiamo solo se Chosen > Rejected + margine
        if score_chosen > (score_rejected + 0.1):
            dataset_buffer.append(entry)
            valid_pairs += 1
        else:
            skipped_pairs += 1
            
    except Exception as e:
        print(f"Errore riga: {e}") 
        continue

Analisi coppie in corso su: /kaggle/input/data-to-judge/dpo_dataset_ready_v2.jsonl
Totale righe da analizzare: 2000


100%|██████████| 2000/2000 [01:20<00:00, 24.77it/s]


In [12]:
print("-" * 30)
print(f"FINITO.")
print(f" Totale Analizzati: {len(lines)}")
print(f" Scartati: {skipped_pairs}")
print(f" TENUTI: {valid_pairs}")

with open(OUTPUT_JSONL, 'w') as f:
    for item in dataset_buffer:
        f.write(json.dumps(item) + "\n")

print(f"\n File salvato: {OUTPUT_JSONL}")

------------------------------
FINITO.
 Totale Analizzati: 2000
 Scartati: 199
 TENUTI: 1801

 File salvato: dpo_dataset_FILTERED.jsonl


In [14]:
# ==========================================
# ANALISI QUALITATIVA SCARTI (PER LA TESI)
# ==========================================
# Cerchiamo 5 esempi dove il Giudice ha detto "NO"
# e stampiamo i dettagli per capire perché.

print(f"CACCIA AGLI ERRORI su: {INPUT_JSONL}")
print("=" * 70)

cattivi_trovati = 0

with open(INPUT_JSONL, 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if cattivi_trovati >= 5: break # Ci fermiamo dopo 5 esempi

    try:
        entry = json.loads(line)
        
        # Gestione formato (lista vs stringa)
        if isinstance(entry['chosen'], list):
            txt_chosen = entry['chosen'][-1]['content']
            txt_rejected = entry['rejected'][-1]['content']
        else:
            txt_chosen = entry['chosen']
            txt_rejected = entry['rejected']
        
        voto_chosen = get_score(txt_chosen)
        voto_rejected = get_score(txt_rejected)
        diff = voto_chosen - voto_rejected
        
        # CRITERIO DI SCARTO:
        # Se la Chosen non supera la Rejected di almeno 0.1, stampiamo l'errore
        if diff <= 0.1:
            cattivi_trovati += 1
            
            print(f"\n COPPIA SCARTATA #{cattivi_trovati} (Riga {i})")
            print("-" * 70)
            print(f"CHOSEN (Originale) | Voto Giudice: {voto_chosen:.2f}")
            print(f"\"{txt_chosen[:150]}...\"") # Tagliamo se troppo lunga
            print("-" * 30)
            print(f"REJECTED (Zephyr)  | Voto Giudice: {voto_rejected:.2f}")
            print(f"\"{txt_rejected[:150]}...\"")
            print("-" * 30)
            
            if voto_rejected > voto_chosen:
                print(f" PROBLEMA GRAVE: La versione 'Rejected' piace di più al giudice! (Diff: {diff:.2f})")
            else:
                print(f" PROBLEMA LIEVE: Differenza troppo sottile, dati ambigui. (Diff: {diff:.2f})")
            print("=" * 70)

    except Exception as e:
        continue

CACCIA AGLI ERRORI su: /kaggle/input/data-to-judge/dpo_dataset_ready_v2.jsonl

 COPPIA SCARTATA #1 (Riga 0)
----------------------------------------------------------------------
CHOSEN (Originale) | Voto Giudice: 5.75
"The fact that the British call math ""maths"" scares me, since the only thing more frightening than math is plural math.
..."
------------------------------
REJECTED (Zephyr)  | Voto Giudice: 5.71
"Serious statement: I find it noteworthy that in the United Kingdom, the subject of mathematics is referred to as'maths,' which causes me concern due t..."
------------------------------
 PROBLEMA LIEVE: Differenza troppo sottile, dati ambigui. (Diff: 0.05)

 COPPIA SCARTATA #2 (Riga 8)
----------------------------------------------------------------------
CHOSEN (Originale) | Voto Giudice: 3.72
"The word politics is derived from two words The word poly meaning many and the word ticks meaning blood sucking parasite.
..."
------------------------------
REJECTED (Zephyr)  | Voto

In [15]:
print("\n" + "=" * 70)
print(" [PARTE 2] ESEMPI DI COPPIE ACCETTATE")
print("-" * 70)

buoni_trovati = 0
for i, line in enumerate(lines):
    if buoni_trovati >= 5: break 

    try:
        entry = json.loads(line)
        # Recupero testo
        if isinstance(entry['chosen'], list):
            txt_c = entry['chosen'][-1]['content']
            txt_r = entry['rejected'][-1]['content']
        else:
            txt_c = entry['chosen']
            txt_r = entry['rejected']
        
        s_c = get_score(txt_c)
        s_r = get_score(txt_r)
        
        # Se ACCETTATO (Chosen > Rejected + margine)
        if s_c > (s_r + 0.1):
            buoni_trovati += 1
            diff = s_c - s_r
            print(f" SUCCESSO #{buoni_trovati} (Riga {i})")
            print(f" Chosen Score:   {s_c:.2f}")
            print(f" Rejected Score: {s_r:.2f}")
            print(f" Margine (Delta): +{diff:.2f}")
            print(f" Chosen: \"{txt_c[:100]}...\"")
            print(f" Reject: \"{txt_r[:100]}...\"")
            print("-" * 40)
    except: continue

print("\n Analisi terminata.")


 [PARTE 2] ESEMPI DI COPPIE ACCETTATE
----------------------------------------------------------------------
 SUCCESSO #1 (Riga 1)
 Chosen Score:   6.42
 Rejected Score: 5.39
 Margine (Delta): +1.03
 Chosen: "Why did Lady Gaga wear seashells to the VMAs? Because B-shells were too small.
..."
 Reject: "Lady Gaga chose to wear seashells as accessories during her appearance at the VMAs because she prefe..."
----------------------------------------
 SUCCESSO #2 (Riga 2)
 Chosen Score:   7.15
 Rejected Score: 6.11
 Margine (Delta): +1.05
 Chosen: "What do you get when you mix alcohol with literature? Tequila Mockingbird!
..."
 Reject: "Serious statement: The combination of alcohol and literature is referred to as 'Tequila Mockingbird,..."
----------------------------------------
 SUCCESSO #3 (Riga 3)
 Chosen Score:   6.70
 Rejected Score: 3.37
 Margine (Delta): +3.33
 Chosen: "Russian Cosmonauts have been banned from telling jokes on the Interational Space Station... ... beca..."
 Reject: 